## Expense Claim Patterns and Fraud Analysis (Flag 90)

### Dataset Overview
The dataset consists of 500 entries from the ServiceNow `fm_expense_line` table. Columns include user, category, opened_at, type, number, processed_date, source_id, state, short_description, and ci. States include Processed (349), Pending (56), Declined (50), and Submitted (45). Categories include Assets (152), Services (134), Travel (116), and Miscellaneous (98).

### Your Objective
**Objective**: Detect and investigate patterns in expense claims by user and category to identify potential anomalies or policy compliance issues.

**Role**: Compliance and Audit Analyst

**Category**: Finance Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks. 

In [1]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from openai import OpenAI
from pandas import date_range



### Load Dataset
This cell loads the expense dataset to be analyzed. The data is orginally saved in the from a CSV file, and is here imported into a DataFrame. The steps involve specifying the path to the dataset, using pandas to read the file, and confirming its successful load by inspecting the first few table entries.

In [2]:
import pandas as pd
dataset_path = "csvs/flag-90.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

,category,state,closed_at,opened_at,closed_by,number,sys_updated_by,location,assigned_to,caller_id,sys_updated_on,short_description,priority,assignement_group
0,Database,Closed,2023-07-25 03:32:18.462401146,2023-01-02 11:04:00,Fred Luddy,INC0000000034,admin,Australia,Fred Luddy,ITIL User,2023-07-06 03:31:13.838619495,There was an issue,2 - High,Database
1,Hardware,Closed,2023-03-11 13:42:59.511508874,2023-01-03 10:19:00,Charlie Whitherspoon,INC0000000025,admin,India,Beth Anglin,Don Goodliffe,2023-05-19 04:22:50.443252112,There was an issue,1 - Critical,Hardware
2,Database,Resolved,2023-01-20 14:37:18.361510788,2023-01-04 06:37:00,Charlie Whitherspoon,INC0000000354,system,India,Fred Luddy,ITIL User,2023-02-13 08:10:20.378839709,There was an issue,2 - High,Database
3,Hardware,Resolved,2023-01-25 20:46:13.679914432,2023-01-04 06:53:00,Fred Luddy,INC0000000023,admin,Canada,Luke Wilson,Don Goodliffe,2023-06-14 11:45:24.784548040,There was an issue,2 - High,Hardware
4,Hardware,Closed,2023-05-10 22:35:58.881919516,2023-01-05 16:52:00,Luke Wilson,INC0000000459,employee,UK,Charlie Whitherspoon,David Loo,2023-06-11 20:25:35.094482408,There was an issue,2 - High,Hardware


### **Question 1: What are the total expenses by department?**

This analysis will help identify which departments are incurring the most significant expenses. By summing up the expenses for each department, we can gain insights into how financial resources are allocated across the organization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_counts = df['category'].value_counts().reset_index()
category_counts.columns = ['category', 'count']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='category', y='count', data=category_counts, palette='Set2')
plt.title('Total Expenses by Category')
plt.xlabel('Category')
plt.ylabel('Number of Expenses')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "comparative",
    "insight": "The expense category distribution shows Assets (152), Services (134), Travel (116), and Miscellaneous (98), with no single category overwhelmingly dominant, suggesting a diverse expense portfolio.",
    "insight_value": {
        "Assets": 152,
        "Services": 134,
        "Travel": 116,
        "Miscellaneous": 98
    },
    "plot": {
        "plot_type": "bar",
        "title": "Total Expenses by Category",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing expense count for each category."
    },
    "question": "What are the total expenses by department?",
    "actionable_insight": "The relatively balanced distribution across categories indicates that expense controls should be applied evenly. No single category should be deprioritized in policy enforcement."
}

### **Question 2:** What are the average expenses per user within each department?

This analysis will reveal the average expense per user within each department. This insight helps to understand individual spending behavior and whether there are significant discrepancies across departments.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

user_counts = df['user'].value_counts().head(10).reset_index()
user_counts.columns = ['user', 'count']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='user', y='count', data=user_counts, palette='muted')
plt.title('Top 10 Users by Number of Expense Submissions')
plt.xlabel('User')
plt.ylabel('Number of Expenses')
plt.xticks(rotation=45, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
{
    "data_type": "comparative",
    "insight": "Some users submit significantly more expense reports than others. Top submitters account for a disproportionate share of all expense reports, warranting review.",
    "insight_value": {
        "Processed": 349,
        "total_users": "multiple",
        "total_expenses": 500
    },
    "plot": {
        "plot_type": "bar",
        "title": "Top 10 Users by Number of Expense Submissions",
        "x_axis": {
            "name": "User"
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing top 10 users by the number of expense reports submitted."
    },
    "question": "What are the average expenses per user within each department?",
    "actionable_insight": "Users with unusually high submission counts should have their expenses audited for policy compliance. Establishing per-user submission thresholds or requiring additional approval for high-volume submitters can improve oversight."
}

### **Question 3:What are the total expenses by category?**


Understanding the distribution of expenses across different categories can help identify areas where the company is spending the most and potentially optimize costs.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='state', y='count', data=state_counts, palette='Set3')
plt.title('Expense State Distribution')
plt.xlabel('State')
plt.ylabel('Number of Expenses')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "categorical",
    "insight": "The 'Processed' state dominates with 69.8% (349 out of 500) of expenses, while the decline rate stands at 10% (50 expenses), indicating a generally healthy but improvable approval workflow.",
    "insight_value": {
        "Processed": 349,
        "Pending": 56,
        "Declined": 50,
        "Submitted": 45,
        "decline_rate": "10%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Expense State Distribution",
        "x_axis": {
            "name": "State"
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing the distribution of expenses across processing states."
    },
    "question": "What are the total expenses by category?",
    "actionable_insight": "With 10% of expenses declined, investigating common decline reasons and providing targeted guidance to submitters can reduce rejections and improve efficiency."
}

### **Question 4:  How many expenses have been processed by each department?**


This analysis reveals the workload and activity level of each department by showing the number of expenses that have been processed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df['opened_at'] = pd.to_datetime(df['opened_at'])
df['processed_date'] = pd.to_datetime(df['processed_date'], errors='coerce')
df['processing_days'] = (df['processed_date'] - df['opened_at']).dt.days

processed = df[df['state'] == 'Processed'].dropna(subset=['processing_days'])

plt.figure(figsize=(10, 6))
sns.boxplot(x='category', y='processing_days', data=processed, palette='Set2')
plt.title('Processing Time by Category for Processed Expenses')
plt.xlabel('Category')
plt.ylabel('Processing Time (days)')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "comparative",
    "insight": "Processing times (opened_at to processed_date) for 'Processed' expenses reveal how quickly expenses move through the system, with potential differences across categories.",
    "insight_value": {
        "Processed": 349,
        "total": 500
    },
    "plot": {
        "plot_type": "boxplot",
        "title": "Processing Time by Category for Processed Expenses",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Processing Time (days)"
        },
        "description": "Box plot showing processing time distributions for each category, limited to Processed expenses."
    },
    "question": "How many expenses have been processes by each department?",
    "actionable_insight": "Categories with longer processing times may benefit from streamlined approval workflows or dedicated reviewers to reduce cycle time."
}

### **Question 5:  What is the average processing time by department?**


This analysis will provide insights into how quickly each department processes expenses, which can highlight potential bottlenecks or efficiency issues.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_state = df.groupby(['category', 'state']).size().unstack(fill_value=0)
if 'Declined' in category_state.columns:
    category_state['decline_rate'] = category_state['Declined'] / category_state.sum(axis=1) * 100
    decline_by_cat = category_state['decline_rate'].reset_index()
    decline_by_cat.columns = ['category', 'decline_rate']
    plt.figure(figsize=(8, 6))
    bar_plot = sns.barplot(x='category', y='decline_rate', data=decline_by_cat, palette='Reds_d')
    plt.title('Decline Rate by Category')
    plt.xlabel('Category')
    plt.ylabel('Decline Rate (%)')
    for p in bar_plot.patches:
        bar_plot.annotate(f'{p.get_height():.1f}%',
                          (p.get_x() + p.get_width() / 2., p.get_height()),
                          ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plt.tight_layout()
    plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "comparative",
    "insight": "The Decline rate varies by category, with some categories potentially showing significantly higher rejection rates, indicating category-specific compliance issues.",
    "insight_value": {
        "Declined": 50,
        "decline_rate": "10%",
        "categories": [
            "Assets",
            "Services",
            "Travel",
            "Miscellaneous"
        ]
    },
    "plot": {
        "plot_type": "bar",
        "title": "Decline Rate by Category",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Decline Rate (%)"
        },
        "description": "Bar chart showing the percentage of declined expenses per category."
    },
    "question": "What is the average processing time by department?",
    "actionable_insight": "Categories with higher decline rates need targeted policy clarification. Employees in those categories should receive training or pre-submission checklists to reduce errors."
}

### Summary of Findings (Flag 90)



1. **Balanced Category Distribution**: Unlike other flags, this dataset has a more evenly distributed category breakdown with Assets (30.4%), Services (26.8%), Travel (23.2%), and Miscellaneous (19.6%).

2. **User Submission Patterns**: Identifying top submitters by volume enables targeted auditing of high-activity users, helping to detect potential policy violations.

3. **Processing State**: 69.8% of expenses are Processed and 10% Declined. Reducing the Pending backlog (56 expenses) and the decline rate would improve overall efficiency.

4. **Category Processing Times**: Analyzing processing times by category for completed expenses reveals workflow bottlenecks in specific expense types.

5. **Category Decline Rates**: Variability in decline rates across categories points to category-specific compliance issues that require targeted educational interventions.